In [2]:
%load_ext autoreload
%autoreload 2

import sys
import torch
import numpy as np
import os
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm # Note: Use tqdm.notebook for pretty UI bars

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data_pipeline import CleanedGreekLettersDataset, VGG16FeatureExtractor

In [3]:
from torchvision import transforms

# 1. Hardware setup (Leveraging the M3 Apple Silicon GPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. VGG-16 standard ImageNet preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATA_DIR = "../data/ALPUB_v2/images"
OUTPUT_DIR = "../data/extracted_features"

# 4. Instantiate Dataset and DataLoader safely
dataset = CleanedGreekLettersDataset(root_dir=DATA_DIR, transform=transform)

assert len(dataset) > 0, f"Critical Error: 0 images found in {DATA_DIR}. Check your folder structure!"

# Because CleanedGreekLettersDataset is imported from an external file, 
# macOS multiprocessing 'spawn' can safely pickle it. num_workers=4 is safe!
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4)

# 5. Initialize Model
model = VGG16FeatureExtractor().to(device)
model.eval() # Freezes Dropout and BatchNorm

# 6. Extraction Loop
all_features = []
all_labels = []

print(f"Starting feature extraction for {len(dataset)} images...")

# torch.no_grad() is critical here to prevent the M3 unified memory from filling
# up with backward-pass computational graphs we don't need.
with torch.no_grad():
    # Using tqdm for a clean progress bar
    for images, labels in tqdm(dataloader, desc="Extracting VGG-16 Features"):
        # Move batch to M3 GPU
        images = images.to(device)
        
        # Forward pass through convolutional base + GAP
        features = model(images)
        
        # Move back to CPU memory and convert to NumPy for scikit-learn
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

# 7. Aggregate and Save
X = np.vstack(all_features)
y = np.concatenate(all_labels)

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.save(os.path.join(OUTPUT_DIR, "X_features_vgg16.npy"), X)
np.save(os.path.join(OUTPUT_DIR, "y_labels.npy"), y)

print(f"Extraction complete! Saved X shape: {X.shape}, y shape: {y.shape}")

Using device: mps
Starting feature extraction for 54616 images...


Extracting VGG-16 Features:   0%|          | 0/427 [00:00<?, ?it/s]

Extraction complete! Saved X shape: (54616, 512), y shape: (54616,)


In [3]:
from src.data_pipeline import EarlyVGG16FeatureExtractor
from torchvision import transforms

# 1. Hardware setup (Leveraging the M3 Apple Silicon GPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. VGG-16 standard ImageNet preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATA_DIR = "../data/ALPUB_v2/images"
OUTPUT_DIR = "../data/extracted_features_early"

# 4. Instantiate Dataset and DataLoader safely
dataset = CleanedGreekLettersDataset(root_dir=DATA_DIR, transform=transform)

assert len(dataset) > 0, f"Critical Error: 0 images found in {DATA_DIR}. Check your folder structure!"

# Because CleanedGreekLettersDataset is imported from an external file, 
# macOS multiprocessing 'spawn' can safely pickle it. num_workers=4 is safe!
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4)

# 5. Initialize Model
model = EarlyVGG16FeatureExtractor().to(device)
model.eval() # Freezes Dropout and BatchNorm

# 6. Extraction Loop
all_features = []
all_labels = []

print(f"Starting feature extraction for {len(dataset)} images...")

# torch.no_grad() is critical here to prevent the M3 unified memory from filling
# up with backward-pass computational graphs we don't need.
with torch.no_grad():
    # Using tqdm for a clean progress bar
    for images, labels in tqdm(dataloader, desc="Extracting VGG-16 Features"):
        # Move batch to M3 GPU
        images = images.to(device)
        
        # Forward pass through convolutional base + GAP
        features = model(images)
        
        # Move back to CPU memory and convert to NumPy for scikit-learn
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

# 7. Aggregate and Save
X = np.vstack(all_features)
y = np.concatenate(all_labels)

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.save(os.path.join(OUTPUT_DIR, "X_features_vgg16.npy"), X)
np.save(os.path.join(OUTPUT_DIR, "y_labels.npy"), y)

print(f"Extraction complete! Saved X shape: {X.shape}, y shape: {y.shape}")

Using device: mps
Starting feature extraction for 54616 images...


Extracting VGG-16 Features:   0%|          | 0/427 [00:00<?, ?it/s]

Extraction complete! Saved X shape: (54616, 256), y shape: (54616,)


In [3]:
from src.data_pipeline import VGG16SPPFeatureExtractor

from torchvision import transforms

# 1. Hardware setup (Leveraging the M3 Apple Silicon GPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. VGG-16 standard ImageNet preprocessing
transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATA_DIR = "../data/ALPUB_v2/images"
OUTPUT_DIR = "../data/extracted_features_SPP"

# 4. Instantiate Dataset and DataLoader safely
dataset = CleanedGreekLettersDataset(root_dir=DATA_DIR, transform=transform)

assert len(dataset) > 0, f"Critical Error: 0 images found in {DATA_DIR}. Check your folder structure!"

# Because CleanedGreekLettersDataset is imported from an external file, 
# macOS multiprocessing 'spawn' can safely pickle it. num_workers=4 is safe!
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4)

# 5. Initialize Model
model = VGG16SPPFeatureExtractor().to(device)
model.eval() # Freezes Dropout and BatchNorm

# 6. Extraction Loop
all_features = []
all_labels = []

print(f"Starting feature extraction for {len(dataset)} images...")

# torch.no_grad() is critical here to prevent the M3 unified memory from filling
# up with backward-pass computational graphs we don't need.
with torch.no_grad():
    # Using tqdm for a clean progress bar
    for images, labels in tqdm(dataloader, desc="Extracting VGG-16 Features"):
        # Move batch to M3 GPU
        images = images.to(device)
        
        # Forward pass through convolutional base + GAP
        features = model(images)
        
        # Move back to CPU memory and convert to NumPy for scikit-learn
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

# 7. Aggregate and Save
X = np.vstack(all_features)
y = np.concatenate(all_labels)

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.save(os.path.join(OUTPUT_DIR, "X_features_vgg16.npy"), X)
np.save(os.path.join(OUTPUT_DIR, "y_labels.npy"), y)

print(f"Extraction complete! Saved X shape: {X.shape}, y shape: {y.shape}")

Using device: mps
Starting feature extraction for 125280 images...


Extracting VGG-16 Features:   0%|          | 0/979 [00:00<?, ?it/s]

Extraction complete! Saved X shape: (125280, 2048), y shape: (125280,)
